In [0]:
from pyspark.sql import functions as F
import re
import unicodedata

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.silver")


def normalize_name(value: str) -> str:
    value = value or ""
    value = unicodedata.normalize("NFKD", value)
    value = value.encode("ascii", "ignore").decode("ascii")
    value = value.upper().strip()
    value = re.sub(r"[^A-Z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")


def normalize_columns(df):
    used = {}

    for old_name in df.columns:
        new_name = normalize_name(old_name)

        if new_name in used:
            used[new_name] += 1
            new_name = f"{new_name}_{used[new_name]}"
        else:
            used[new_name] = 1

        if old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)

    return df


def first_existing(df, candidates):
    normalized_candidates = [normalize_name(c) for c in candidates]

    for candidate in normalized_candidates:
        if candidate in df.columns:
            return F.col(candidate)

    return F.lit(None)


def clean_text(col):
    return F.trim(col.cast("string"))


def clean_upper(col):
    return F.upper(F.trim(col.cast("string")))


def clean_digits(col):
    return F.regexp_replace(col.cast("string"), r"[^0-9]", "")

In [0]:
firme_raw = normalize_columns(spark.table("company_ro.bronze.onrc_firme_raw"))
caen_raw = normalize_columns(spark.table("company_ro.bronze.onrc_caen_autorizat_raw"))

stare_raw = normalize_columns(spark.table("company_ro.bronze.onrc_stare_firma_raw"))
n_stare_raw = normalize_columns(spark.table("company_ro.bronze.n_stare_firma_raw"))

print("firme_raw columns:")
print(firme_raw.columns)

print("caen_raw columns:")
print(caen_raw.columns)

print("stare_raw columns:")
print(stare_raw.columns)

print("n_stare_raw columns:")
print(n_stare_raw.columns)

In [0]:
stare_firma_bridge = (
    stare_raw
    .select(
        clean_text(first_existing(stare_raw, [
            "COD_INMATRICULARE",
            "NR_INMATRICULARE",
            "NUMAR_INMATRICULARE"
        ])).alias("cod_inmatriculare"),

        clean_digits(first_existing(stare_raw, [
            "COD",
            "COD_STARE",
            "COD_STARE_FIRMA"
        ])).alias("cod_stare_firma")
    )
    .filter(F.col("cod_inmatriculare").isNotNull())
    .filter(F.col("cod_inmatriculare") != "")
    .dropDuplicates(["cod_inmatriculare"])
)

stare_firma_lookup = (
    n_stare_raw
    .select(
        clean_digits(first_existing(n_stare_raw, [
            "COD",
            "COD_STARE",
            "COD_STARE_FIRMA"
        ])).alias("cod_stare_firma"),

        clean_text(first_existing(n_stare_raw, [
            "DENUMIRE",
            "DENUMIRE_STARE",
            "STARE_FIRMA"
        ])).alias("stare_firma")
    )
    .filter(F.col("cod_stare_firma").isNotNull())
    .filter(F.col("cod_stare_firma") != "")
    .dropDuplicates(["cod_stare_firma"])
)

firme_base = (
    firme_raw
    .select(
        clean_text(first_existing(firme_raw, [
            "COD_INMATRICULARE",
            "NR_INMATRICULARE",
            "NUMAR_INMATRICULARE"
        ])).alias("cod_inmatriculare"),

        clean_digits(first_existing(firme_raw, [
            "CUI",
            "COD_FISCAL",
            "CIF",
            "COD_UNIC_INREGISTRARE"
        ])).alias("cui"),

        clean_text(first_existing(firme_raw, [
            "DENUMIRE",
            "DENUMIRE_FIRMA",
            "NUME_FIRMA"
        ])).alias("denumire"),

        clean_upper(first_existing(firme_raw, [
            "JUDET",
            "DEN_JUDET",
            "DENUMIRE_JUDET",
            "ADR_JUDET",
            "ADR_DEN_JUDET",
            "ADRESA_JUDET"
        ])).alias("judet"),

        clean_upper(first_existing(firme_raw, [
            "LOCALITATE",
            "DEN_LOCALITATE",
            "DENUMIRE_LOCALITATE",
            "ORAS",
            "MUNICIPIU",
            "ADR_LOCALITATE",
            "ADR_DEN_LOCALITATE",
            "ADRESA_LOCALITATE",
            "ADR_ORAS",
            "ADR_MUNICIPIU"
        ])).alias("localitate"),

        clean_text(first_existing(firme_raw, [
            "ADRESA",
            "ADRESA_COMPLETA",
            "SEDIU",
            "ADR_ADRESA",
            "ADR_STRADA",
            "ADR_DEN_STRADA"
        ])).alias("adresa"),

        clean_text(first_existing(firme_raw, [
            "FORMA_JURIDICA",
            "DEN_FORMA_JURIDICA",
            "FORM_JURIDICA",
            "COD_FORMA_JURIDICA"
        ])).alias("forma_juridica"),

        first_existing(firme_raw, [
            "_ingested_at",
            "INGESTED_AT"
        ]).alias("_ingested_at"),

        first_existing(firme_raw, [
            "_source_file",
            "SOURCE_FILE"
        ]).alias("_source_file")
    )
    .dropDuplicates(["cod_inmatriculare"])
)

dim_firma = (
    firme_base
    .join(stare_firma_bridge, "cod_inmatriculare", "left")
    .join(stare_firma_lookup, "cod_stare_firma", "left")
    .withColumn(
        "an_infiintare",
        F.regexp_extract(F.col("cod_inmatriculare"), r"(\d{4})$", 1).cast("int")
    )
    .withColumn(
        "is_activa",
        F.when(F.lower(F.col("stare_firma")) == "funcțiune", F.lit(True))
         .when(F.lower(F.col("stare_firma")) == "functiune", F.lit(True))
         .otherwise(F.lit(False))
    )
    .withColumn(
        "firma_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("cod_inmatriculare"), F.lit("")),
                F.coalesce(F.col("cui"), F.lit(""))
            ),
            256
        )
    )
    .select(
        "firma_key",
        "cod_inmatriculare",
        "cui",
        "denumire",
        "judet",
        "localitate",
        "adresa",
        "cod_stare_firma",
        "stare_firma",
        "is_activa",
        "forma_juridica",
        "an_infiintare",
        "_ingested_at",
        "_source_file"
    )
)

display(dim_firma.limit(20))
print("Rows:", dim_firma.count())
print("Columns:", dim_firma.columns)

In [0]:
(
    dim_firma
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("company_ro.silver.dim_firma")
)

In [0]:
bridge_firma_caen = (
    caen_raw
    .select(
        clean_text(first_existing(caen_raw, [
            "COD_INMATRICULARE",
            "NR_INMATRICULARE",
            "NUMAR_INMATRICULARE"
        ])).alias("cod_inmatriculare"),

        clean_digits(first_existing(caen_raw, [
            "COD_CAEN_AUTORIZAT",
            "COD_CAEN",
            "CAEN"
        ])).alias("cod_caen"),

        clean_text(first_existing(caen_raw, [
            "VER_CAEN_AUTORIZAT",
            "VERSIUNE_CAEN",
            "VER_CAEN"
        ])).alias("versiune_caen"),

        first_existing(caen_raw, [
            "_ingested_at",
            "INGESTED_AT"
        ]).alias("_ingested_at"),

        first_existing(caen_raw, [
            "_source_file",
            "SOURCE_FILE"
        ]).alias("_source_file")
    )
    .filter(F.col("cod_inmatriculare").isNotNull())
    .filter(F.col("cod_caen").isNotNull())
    .filter(F.col("cod_caen") != "")
    .withColumn(
        "firma_caen_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("cod_inmatriculare"), F.lit("")),
                F.coalesce(F.col("cod_caen"), F.lit("")),
                F.coalesce(F.col("versiune_caen"), F.lit(""))
            ),
            256
        )
    )
    .dropDuplicates(["cod_inmatriculare", "cod_caen", "versiune_caen"])
)

display(bridge_firma_caen.limit(20))
print("Rows:", bridge_firma_caen.count())
print("Columns:", bridge_firma_caen.columns)

In [0]:
(
    bridge_firma_caen
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("company_ro.silver.bridge_firma_caen")
)

In [0]:
display(spark.sql("SHOW TABLES IN company_ro.silver"))

In [0]:
display(spark.sql("""
SELECT 'dim_firma' AS table_name, COUNT(*) AS rows
FROM company_ro.silver.dim_firma

UNION ALL

SELECT 'bridge_firma_caen', COUNT(*)
FROM company_ro.silver.bridge_firma_caen

UNION ALL

SELECT 'dim_caen', COUNT(*)
FROM company_ro.silver.dim_caen
"""))

In [0]:
# Data Quality Check 1: Bridge table - cod_inmatriculare in dim_firma
print("=== DQ Check 1: Bridge FK Validation (cod_inmatriculare) ===")

bridge_firma_check = spark.sql("""
SELECT 
  COUNT(DISTINCT b.cod_inmatriculare) AS total_bridge_companies,
  COUNT(DISTINCT d.cod_inmatriculare) AS companies_with_dim_match,
  COUNT(DISTINCT b.cod_inmatriculare) - COUNT(DISTINCT d.cod_inmatriculare) AS orphaned_companies
FROM company_ro.silver.bridge_firma_caen b
LEFT JOIN company_ro.silver.dim_firma d
  ON b.cod_inmatriculare = d.cod_inmatriculare
""")

result = bridge_firma_check.collect()[0]
total = result['total_bridge_companies']
matched = result['companies_with_dim_match']
orphaned = result['orphaned_companies']
match_rate = (matched / total * 100) if total > 0 else 0

print(f"Total companies in bridge: {total:,}")
print(f"Companies matching dim_firma: {matched:,}")
print(f"Orphaned companies: {orphaned:,}")
print(f"Match rate: {match_rate:.2f}%")

if match_rate < 95:
    print(f"⚠ WARNING: Only {match_rate:.2f}% of bridge companies have matching firm dimensions")
else:
    print(f"✓ Acceptable: {match_rate:.2f}% company match rate")

In [0]:
# Data Quality Check 2: Bridge table - cod_caen in dim_caen
print("\n=== DQ Check 2: Bridge FK Validation (cod_caen) ===")

bridge_caen_check = spark.sql("""
SELECT 
  COUNT(*) AS total_bridge_rows,
  COUNT(DISTINCT b.cod_caen) AS distinct_caen_in_bridge,
  COUNT(DISTINCT CASE 
    WHEN d.cod_caen IS NOT NULL THEN b.cod_caen 
  END) AS caen_codes_with_dim_match,
  COUNT(DISTINCT CASE 
    WHEN d.cod_caen IS NULL AND b.cod_caen IS NOT NULL THEN b.cod_caen 
  END) AS orphaned_caen_codes
FROM company_ro.silver.bridge_firma_caen b
LEFT JOIN company_ro.silver.dim_caen d
  ON LPAD(b.cod_caen, 4, '0') = d.cod_caen
""")

result = bridge_caen_check.collect()[0]
total_rows = result['total_bridge_rows']
distinct_caen = result['distinct_caen_in_bridge']
matched_caen = result['caen_codes_with_dim_match']
orphaned_caen = result['orphaned_caen_codes']
match_rate = (matched_caen / distinct_caen * 100) if distinct_caen > 0 else 0

print(f"Total bridge rows: {total_rows:,}")
print(f"Distinct CAEN codes in bridge: {distinct_caen:,}")
print(f"CAEN codes matching dim_caen: {matched_caen:,}")
print(f"Orphaned CAEN codes: {orphaned_caen:,}")
print(f"Match rate: {match_rate:.2f}%")

if match_rate < 90:
    print(f"⚠ WARNING: Only {match_rate:.2f}% of bridge CAEN codes have matching dimensions")
else:
    print(f"✓ Acceptable: {match_rate:.2f}% CAEN match rate")

In [0]:
# Data Quality Check 3: Dimension key uniqueness
print("\n=== DQ Check 3: Dimension Key Uniqueness ===")

# Check dim_firma for duplicate keys
firma_dups = spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT firma_key) AS distinct_keys,
  COUNT(DISTINCT cui) AS distinct_cui,
  COUNT(DISTINCT cod_inmatriculare) AS distinct_cod_inmatriculare
FROM company_ro.silver.dim_firma
""")

result = firma_dups.collect()[0]
firma_total = result['total_rows']
firma_keys = result['distinct_keys']
firma_cui = result['distinct_cui']
firma_cod = result['distinct_cod_inmatriculare']

print(f"\ndim_firma:")
print(f"  Total rows: {firma_total:,}")
print(f"  Distinct firma_key: {firma_keys:,}")
print(f"  Distinct CUI: {firma_cui:,}")
print(f"  Distinct cod_inmatriculare: {firma_cod:,}")

if firma_total != firma_keys:
    print(f"  ⚠ WARNING: Duplicate firma_keys detected ({firma_total - firma_keys} duplicates)")
else:
    print(f"  ✓ firma_key is unique")

# Check dim_caen for duplicate keys
caen_dups = spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT caen_key) AS distinct_keys,
  COUNT(DISTINCT cod_caen) AS distinct_cod_caen
FROM company_ro.silver.dim_caen
""")

result = caen_dups.collect()[0]
caen_total = result['total_rows']
caen_keys = result['distinct_keys']
caen_cod = result['distinct_cod_caen']

print(f"\ndim_caen:")
print(f"  Total rows: {caen_total:,}")
print(f"  Distinct caen_key: {caen_keys:,}")
print(f"  Distinct cod_caen: {caen_cod:,}")

if caen_total != caen_keys:
    print(f"  ⚠ WARNING: Duplicate caen_keys detected ({caen_total - caen_keys} duplicates)")
else:
    print(f"  ✓ caen_key is unique")

if caen_total != caen_cod:
    print(f"  ⚠ WARNING: Duplicate cod_caen detected ({caen_total - caen_cod} duplicates)")
else:
    print(f"  ✓ cod_caen is unique")

In [0]:
display(spark.sql("""
SELECT
  COUNT(*) AS bridge_rows,
  COUNT(f.cod_inmatriculare) AS rows_matching_dim_firma,
  COUNT(c.cod_caen) AS rows_matching_dim_caen
FROM company_ro.silver.bridge_firma_caen b
LEFT JOIN company_ro.silver.dim_firma f
  ON b.cod_inmatriculare = f.cod_inmatriculare
LEFT JOIN company_ro.silver.dim_caen c
  ON b.cod_caen = c.cod_caen
"""))

In [0]:
for c in firme_raw.columns:
    if any(x in c for x in ["JUD", "LOC", "ORAS", "MUN", "ADR"]):
        print(c)